# SPORE_heavy: Systematic Preprocessing and Optimization for Robust Evaluation
**Engineered for Massive-Scale Single-Cell Perturbation Data**

**Overview:** SPORE_heavy is a highly optimized, memory-safe computational pipeline designed to ingest, triage, and normalize massive-scale single-cell RNA sequencing (scRNA-seq) perturbation datasets (e.g., CRISPRa/i screens). The pipeline isolates true perturbation-driven transcriptional responses by systematically eliminating technical artifacts, suppressing generic stress confounders, and performing rigorous data stratification. 

**Architecture:**
The pipeline enforces a strict $O(1)$ memory footprint where possible, utilizing in-place C-buffer mutations, chunked sparse matrix conversions, and randomized solvers to prevent Out-Of-Memory (OOM) failures on datasets exceeding 1.5 million cells. The final output is a biologically concentrated, confounder-free `.h5ad` matrix explicitly formatted for downstream causal inference to construct a Gene Regulatory Network (GRN), which subsequently provides the foundational topology for Graph Neural Network (GNN) training and simulation.

---
## Setup & Pipeline Configuration
Initialization of the computational environment, garbage collection protocols, and the centralized SPORE logger. All hyperparameters, file paths, and dataset-specific metadata are dynamically loaded from the `spore_config.yaml` file to ensure the pipeline remains strictly dataset-agnostic.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Ensure src/ is importable
sys.path.insert(0, '.')

from src.utils import (load_config, setup_logger, snapshot,
                        ensure_sparse, log_memory, force_gc)
from src.plotting import apply_spore_style
from src import plotting as splt
from src import phase0_sparse_convert as p0
from src import phase1_cell_triage as p1
from src import phase2_escaper_filtering as p2
from src import phase3_gene_triage as p3
from src import phase4_splits as p4
from src import phase5_hvg as p5
from src import phase6_normalization as p6
from src import phase7_confounders as p7
from src import phase8_metacells as p8

# Load config and initialize
cfg = load_config('spore_config.yaml')
logger = setup_logger(cfg)
apply_spore_style(cfg)

# Pipeline cell-count tracker for final summary
pipeline_tracker = {}

print(f"SPORE initialized")
print(f"  Dataset : {cfg['dataset']['name']}")
print(f"  Organism: {cfg['dataset']['organism']}")
print(f"  Workers : {cfg['runtime']['n_jobs']}")
log_memory(logger, 'startup')

---
## Phase 0: Data Ingestion & Sparsification
**Objective:** Prevent catastrophic memory fragmentation during initial data loading.
Raw single-cell matrices are often stored in dense arrays that exceed physical RAM limits when inflated. Phase 0 operates as a caching and sanitization firewall. It reads the raw data in backed mode from the disk, safely converts it to a Compressed Sparse Row (CSR) format in localized memory chunks, and caches the sparse matrix. This ensures downstream functions interact exclusively with highly efficient sparse representations.

In [ ]:
import src.phase0_sparse_convert as p0
from src.utils import snapshot, log_memory

# ── RUN PHASE 0 ───────────────────────────────────────────────────────────
adata = p0.run_phase0(cfg, logger)

log_memory(logger, 'post Phase 0')
snapshot(adata, 'Raw data loaded & sparsified', logger)
pipeline_tracker['Raw'] = adata.n_obs

print(f"\n.obs columns: {list(adata.obs.columns)}")
print(f".var columns: {list(adata.var.columns)}")
print(f"\nPerturbation column ('{cfg['dataset']['perturbation_col']}'):")
print(adata.obs[cfg['dataset']['perturbation_col']].value_counts().head(10))

---
## Phase 1: Cell-Level Triage
**Objective:** Eliminate dead cells, low-quality droplets, and generic biological noise.
This phase enforces strict quality control thresholds on a per-cell basis. Cells exhibiting high mitochondrial fractions (indicating ruptured membranes or apoptosis) and cells with extreme UMI/gene count anomalies (indicating empty droplets or multiplets) are discarded. Additionally, universally expressed ribosomal features are stripped from the feature space to prevent them from acting as uninformative dictating hubs in downstream GRN topology.

| Filter | Purpose |
|--------|---------|
| MT% threshold | Remove ruptured cells (leaky mitochondria) |
| Min/Max genes | Remove empty droplets and doublets |
| Min/Max UMI counts | Library size sanity bounds |
| Ribosomal gene removal | Strip RPL/RPS columns that drown perturbation signal |

### 1.1 · Compute QC Metrics & Pre-Filter Visualization

In [ ]:
# MT% vs Total Counts scatter (auto-downsampled)
splt.plot_mt_scatter(adata, cfg)

### 1.2 · Apply Filters

In [ ]:
# Apply all Phase 1 filters (single safe_subset — no OOM)
adata, p1_waterfall = p1.filter_cells(adata, cfg, logger)
pipeline_tracker['Phase 1'] = adata.n_obs
log_memory(logger, 'post Phase 1')

In [ ]:
# Post-filter QC distributions
splt.plot_qc_violin(adata, cfg, phase_label='post')

In [ ]:
# Filtering waterfall
splt.plot_filtering_waterfall(p1_waterfall, cfg, phase='phase1')

---
## Phase 2: Escaper Filtering
**Objective:** Enforce perturbation verification and reclassify non-responsive cells.
In pooled CRISPR screens, cells may receive a perturbation vector but fail to exhibit the expected phenotypic or transcriptional knockdown (escapers). Phase 2 statistically evaluates the target gene's expression against an unperturbed control distribution. Cells identified as escapers are either reclassified or discarded to ensure the downstream models only learn from true causal perturbations.

In [ ]:
# Run escaper filtering (parallelized) + perturbation triage
adata, escaper_stats, pert_sizes = p2.run_phase2(adata, cfg, logger)
pipeline_tracker['Phase 2'] = adata.n_obs
log_memory(logger, 'post Phase 2')

In [ ]:
# Escaper summary (top 40 perturbations by escaper count)
splt.plot_escaper_summary(escaper_stats, cfg)

In [ ]:
# Cells-per-perturbation histogram
splt.plot_perturbation_sizes(
    pert_sizes, cfg,
    min_thresh=cfg['phase2_escaper_filtering']['min_cells_per_perturbation']
)

---
## Phase 3: Gene-Level Triage
**Objective:** Compress the feature space while explicitly shielding causal roots.
To concentrate statistical power and reduce matrix sparsity, Phase 3 evaluates expression frequencies across the entire dataset. Genes falling below scale-adaptive UMI thresholds are pruned. However, a structural "diplomatic immunity" protocol ensures that all explicit perturbation target genes and required downstream computational markers (e.g., cell cycle genes) are rescued from this thresholding, ensuring the causal roots of the network are perfectly preserved.

| Filter | Mode |
|--------|------|
| Ambient RNA | Genes in < 10 cells → dropped |
| Scale-adaptive | Large dataset → mean UMI ≥ 0.25 |
| Target rescue | Perturbation targets force-retained |

In [ ]:
# Run gene-level triage
adata, p3_waterfall, p3_rescued = p3.run_phase3(adata, cfg, logger)
pipeline_tracker['Phase 3'] = adata.n_obs
log_memory(logger, 'post Phase 3')

In [ ]:
# Post-filter gene expression histogram
splt.plot_gene_expression_hist(adata, cfg, phase_label='post')

In [ ]:
# Gene filtering waterfall
splt.plot_filtering_waterfall(p3_waterfall, cfg, phase='phase3_genes')

In [ ]:
# Rescued perturbation targets
if p3_rescued:
    print(f"Rescued {len(p3_rescued)} perturbation targets from gene-level filtering")
    splt.plot_rescued_genes(p3_rescued, adata.n_vars, cfg)
else:
    print("No perturbation targets required rescue in Phase 3")

---
## Phase 4: Data Stratification & Splitting
**Objective:** Establish rigorous data partitions to prevent downstream target leakage.
The filtered matrix is divided into independent Training, Validation, and Test splits. Depending on the configuration, this is executed via zero-shot stratification (isolating entirely distinct perturbations into the test set to evaluate model generalizability) or cell-wise splitting. The resulting splits are serialized to disk to create a safe checkpoint.

In [ ]:
# Run Phase 4 splits
all_splits = p4.run_phase4(adata, cfg, logger)

# Free the pre-split adata — we only need the splits from here
del adata
force_gc(logger)
log_memory(logger, 'after freeing pre-split adata')

# Use the first (or only) split going forward
splits = all_splits[0]
pipeline_tracker['Phase 4 (train)'] = splits['split_info']['train']

In [ ]:
# Split proportions visualization
splt.plot_split_summary(splits['split_info'], cfg)

In [ ]:
# DEG/shift stratification histogram (zero-shot mode only)
if 'deg_counts' in splits:
    splt.plot_deg_stratification(
        splits['deg_counts'],
        cfg['phase4_splits']['zero_shot']['stratify_bins'],
        cfg
    )

---
## Phase 5: Dimensionality Reduction (HVG Selection)
**Objective:** Isolate highly variable biological signal and establish feature ghosting.
To constrain the downstream GRN complexity, the feature space is restricted to the top Highly Variable Genes (HVGs). To ensure downstream confounder mitigation (Phase 7) functions correctly without permanently cluttering the final GRN feature space, this phase initiates "Feature Ghosting." Critical covariates that fail HVG selection are temporarily appended to the matrix as "ghosts" to allow for accurate mathematical regression later.

In [ ]:
# Run HVG selection
hvg_genes, p5_rescued = p5.run_phase5(splits, cfg, logger)
log_memory(logger, 'post Phase 5')

In [ ]:
# HVG variance scatter
splt.plot_hvg_variance(splits['train'], cfg)

In [ ]:
# Rescued targets in HVG step
if p5_rescued:
    print(f"Phase 5 rescued {len(p5_rescued)} perturbation targets")
    splt.plot_rescued_genes(p5_rescued, len(hvg_genes), cfg)
else:
    print("All perturbation targets passed HVG selection")

print(f"\nFinal feature space: {splits['train'].n_vars:,} genes")

---
## Phase 6: Normalization & Scaling
**Objective:** Stabilize variance and normalize sequencing depth across the dataset.
The filtered splits undergo global library size normalization (e.g., scaling to the median total counts) followed by a logarithmic transformation ($\log(1+x)$). To preserve the memory-efficient CSR architecture, manual float32 C-buffer casting is employed to explicitly bypass implicit dense upcasting during Scanpy's mathematical operations.

In [ ]:
# Normalize all splits
raw_sample, norm_sample = p6.run_phase6(splits, cfg, logger)

In [ ]:
# Before/after normalization histograms
splt.plot_normalization_comparison(raw_sample, norm_sample, cfg)

---
## Phase 7: Confounder Mitigation & Feature Pruning
**Objective:** Suppress latent covariates and finalize the matrix for GRN ingestion.
The pipeline applies advanced mitigation techniques to suppress technical and biological confounders. Cell cycle phases are scored and regressed out using the temporarily preserved "ghost" genes. Next, batch effect correction (e.g., Harmony) is applied to align disparate sequencing runs, utilizing a custom $O(1)$ scaling algorithm to prevent dense memory allocation. Finally, the "ghosted" features are permanently pruned, returning a perfectly aligned, dimensionally rigid matrix ready for model training.

In [ ]:
# Run confounder mitigation
splits = p7.run_phase7(splits, cfg, logger)
log_memory(logger, 'post Phase 7')

In [ ]:
# Cell cycle phase distribution
splt.plot_cell_cycle_scores(splits['train'], cfg)

---
## Phase 8: Meta-Cell Aggregation & Systema Calibration (Optional)
**Objective:** Combat dropout sparsity and define the systematic variation geometry.
To bridge the gap between noisy single-cell transcriptomics and deterministic causal inference, this phase groups transcriptionally similar cells into pseudobulked "meta-cells" strictly within their interventional boundaries. This densifies the signal for downstream Graph Neural Networks. 

In [ ]:
# Run Meta-Cell Aggregation
splits = p8.run_phase8(splits, cfg, logger)
log_memory(logger, 'post Phase 8')

---
## Save Final Outputs
*GRN-ready `.h5ad` files for downstream PSGRN*

In [ ]:
# ── SAVE FINAL DATASETS TO DISK ───────────────────────────────────────────

# Inspect the actual data structure to determine if Phase 8 was run
is_metacell = 'n_cells_in_metacell' in splits['train'].obs.columns

for key in ["train", "val", "test"]:
    # Dynamically assign the correct suffix based on the data's physical structure
    suffix = "_metacell.h5ad" if is_metacell else "_singlecell.h5ad"
    
    save_path = f"{cfg['paths']['processed_dir']}/{cfg['dataset']['name']}_{key}{suffix}"
    
    logger.info(f"Saving {key} split to {save_path} ...")
    splits[key].write_h5ad(save_path)

logger.info("All files successfully saved to disk.")

---
## Pipeline Summary

In [ ]:
# Final summary
splt.plot_pipeline_summary(pipeline_tracker, cfg)

print("\n" + "═" * 55)
print("  SPORE Pipeline Complete")
print("═" * 55)
start = list(pipeline_tracker.values())[0]
for phase, count in pipeline_tracker.items():
    pct = count / start * 100
    print(f"  {phase:<25s} {count:>10,} cells  ({pct:5.1f}%)")
print("═" * 55)
print(f"  Final gene features: {splits['train'].n_vars:,}")
print(f"  Split mode: {cfg['phase4_splits']['mode']}")
print(f"  Downstream GRN: {cfg['downstream']['grn']}")
print("═" * 55)
log_memory(logger, 'pipeline complete')